# Transfer Learning (전이학습)

## 특성 추출 기법 : Convolutional Layer는 그대로 가져오고, Fully Connected Layer만 새로 학습

In [1]:
import os
import time
import copy
import glob
import cv2
import shutil
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

## 데이터 가져오기

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

folder_path = '/content/drive/My Drive/data/catanddog'

# Check if the folder exists
if os.path.exists(folder_path):
    print(f"Contents of '{folder_path}':")
    for item in os.listdir(folder_path):
        print(item)
else:
    print(f"Error: Folder '{folder_path}' not found. Please ensure the path is correct and Drive is mounted.")

Contents of '/content/drive/My Drive/data/catanddog':
test
train


#### Random : 데이터를 증강하는 데에 사용 -> Test에서는 사용 안함

In [4]:
transform = transforms.Compose([
    transforms.Resize([256, 256]),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

In [5]:
train_dataset = torchvision.datasets.ImageFolder(root='/content/drive/My Drive/data/catanddog/train', transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, num_workers=8, shuffle=True)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## 모델 가져오기

### 사전 훈련된 ResNet18 모델 사용

In [6]:
resnet18 = models.resnet18(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 138MB/s]


## 특성 추출 기법
마지막 layer만 학습

In [7]:
def set_parameter_requires_grad(model, feature_extracting=True):
    if feature_extracting:
        for param in model.parameters():
            param.requires_grad = False

In [8]:
set_parameter_requires_grad(resnet18)

In [9]:
resnet18.fc = nn.Linear(512, 2)

### 모델의 파라미터 값들을 확인해 보자.

In [10]:
for name, param in resnet18.named_parameters():
  if param.requires_grad:
    print(name, param.data)

fc.weight tensor([[-0.0189,  0.0082,  0.0161,  ..., -0.0217, -0.0274, -0.0090],
        [-0.0130, -0.0223, -0.0190,  ...,  0.0402, -0.0192,  0.0382]])
fc.bias tensor([-0.0002, -0.0258])


## 학습

### 특성 추출

In [11]:
def train_model(model, dataloaders, criterion, optimizer, device, num_epochs=13, is_train=True):
  since = time.time()
  acc_history = []
  loss_history = []
  best_acc = 0.0
  model.to(device) # Move model to device once at the beginning

  for epoch in range(num_epochs):
    print(f'Epoch {epoch}/{num_epochs - 1}')
    print('-' * 10)

    # Each epoch has a training phase
    model.train() # Set model to training mode
    running_loss = 0.0
    running_corrects = 0

    # Iterate over data.
    for inputs, labels in dataloaders:
      inputs = inputs.to(device)
      labels = labels.to(device)

      optimizer.zero_grad()

      # forward
      outputs = model(inputs)
      loss = criterion(outputs, labels)

      _, preds = torch.max(outputs, 1)

      # backward + optimize only if in training phase
      loss.backward()
      optimizer.step()

      # statistics
      running_loss += loss.item() * inputs.size(0)
      running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(dataloaders.dataset)
    epoch_acc = running_corrects.double() / len(dataloaders.dataset)

    print(f'Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

    if epoch_acc > best_acc:
      best_acc = epoch_acc

    acc_history.append(epoch_acc.item())
    loss_history.append(epoch_loss)

    # Save model state after each epoch (correct path and naming)
    save_path = os.path.join('/tmp/', f'model_epoch_{epoch}.pth') # Changed path to /tmp/ and used f-string for unique filenames
    try:
        torch.save(model.state_dict(), save_path)
        print(f"Model for epoch {epoch} saved to {save_path}")
    except Exception as e:
        print(f"Failed to save model for epoch {epoch} to {save_path}: {e}")

  time_elapsed = time.time() - since
  print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
  print(f'Best val Acc: {best_acc:4f}')
  return acc_history, loss_history

### 완전연결층은 학습을 하도록 설정
● 학습을 통해 얻어지는 파라미터를 옵티마이저에 전달해서 최종적으로 모델 학습에 사용

In [12]:
params_to_update = []
for name, param in resnet18.named_parameters():
  if param.requires_grad == True:
    params_to_update.append(param)
    print("\t", name)

optimizer = optim.Adam(params_to_update)

	 fc.weight
	 fc.bias


### 모델 학습

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = nn.CrossEntropyLoss()
train_acc_hist, train_loss_hist = train_model(resnet18, train_loader, criterion, optimizer, device)

Epoch 0/12
----------


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Loss: 0.6769 Acc: 0.5818
Model for epoch 0 saved to /tmp/model_epoch_0.pth
Epoch 1/12
----------
Loss: 0.4299 Acc: 0.8078
Model for epoch 1 saved to /tmp/model_epoch_1.pth
Epoch 2/12
----------
Loss: 0.3715 Acc: 0.8364
Model for epoch 2 saved to /tmp/model_epoch_2.pth
Epoch 3/12
----------
Loss: 0.2744 Acc: 0.9091
Model for epoch 3 saved to /tmp/model_epoch_3.pth
Epoch 4/12
----------
Loss: 0.2594 Acc: 0.9117
Model for epoch 4 saved to /tmp/model_epoch_4.pth
Epoch 5/12
----------
Loss: 0.2547 Acc: 0.9195
Model for epoch 5 saved to /tmp/model_epoch_5.pth
Epoch 6/12
----------
Loss: 0.2371 Acc: 0.9169
Model for epoch 6 saved to /tmp/model_epoch_6.pth
Epoch 7/12
----------
Loss: 0.2641 Acc: 0.8909
Model for epoch 7 saved to /tmp/model_epoch_7.pth
Epoch 8/12
----------
Loss: 0.2925 Acc: 0.8727
Model for epoch 8 saved to /tmp/model_epoch_8.pth
Epoch 9/12
----------
Loss: 0.1904 Acc: 0.9377
Model for epoch 9 saved to /tmp/model_epoch_9.pth
Epoch 10/12
----------
Loss: 0.1763 Acc: 0.9506
Mode

## 테스트

### Test에서는 Random 사용안함

In [14]:
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor()
])
test_dataset = torchvision.datasets.ImageFolder(root='/content/drive/My Drive/data/catanddog/test', transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, num_workers=1, shuffle=True)

### 테스트 평가 함수

In [17]:
def eval_model(model, dataloader, device):
  since = time.time()
  acc_history = []
  best_acc = 0.0
  saved_models = glob.glob('/tmp/*.pth')
  saved_models.sort()
  print('saved model', saved_models)

  for model_path in saved_models:
    print('loading model', model_path)
    model.load_state_dict(torch.load(model_path))
    model.eval()         # dropout 기능 끔
    model.to(device)
    running_corrects = 0

    for inputs, labels in dataloader:
      inputs = inputs.to(device)
      labels = labels.to(device)

      with torch.no_grad():      # 빠른 계산을 도움
        outputs = model(inputs)

      _, preds = torch.max(outputs.data, 1)
      # Removed: preds[preds >= 0.5] = 1
      # Removed: preds[preds < 0.5] = 0
      running_corrects += preds.eq(labels).int().sum() # Changed labels.cpu() to labels

  epoch_acc = running_corrects.double() / len(dataloader.dataset)
  print(f'Acc: {epoch_acc:.4f}')

  if epoch_acc > best_acc:
    best_acc = epoch_acc
    acc_history.append(best_acc.item())

  time_elapsed = time.time() - since
  print(f'Validation complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
  print('Best Acc : {:4f}'.format(best_acc))

  return acc_history

In [18]:
val_acc_hist = eval_model(resnet18, test_loader, device)

saved model ['/tmp/model_epoch_0.pth', '/tmp/model_epoch_1.pth', '/tmp/model_epoch_10.pth', '/tmp/model_epoch_11.pth', '/tmp/model_epoch_12.pth', '/tmp/model_epoch_2.pth', '/tmp/model_epoch_3.pth', '/tmp/model_epoch_4.pth', '/tmp/model_epoch_5.pth', '/tmp/model_epoch_6.pth', '/tmp/model_epoch_7.pth', '/tmp/model_epoch_8.pth', '/tmp/model_epoch_9.pth']
loading model /tmp/model_epoch_0.pth
loading model /tmp/model_epoch_1.pth
loading model /tmp/model_epoch_10.pth
loading model /tmp/model_epoch_11.pth
loading model /tmp/model_epoch_12.pth
loading model /tmp/model_epoch_2.pth
loading model /tmp/model_epoch_3.pth
loading model /tmp/model_epoch_4.pth
loading model /tmp/model_epoch_5.pth
loading model /tmp/model_epoch_6.pth
loading model /tmp/model_epoch_7.pth
loading model /tmp/model_epoch_8.pth
loading model /tmp/model_epoch_9.pth
Acc: 0.9490
Validation complete in 0m 42s
Best Acc : 0.948980
